# Remodelación y Limpieza de datos. 
Raíz de que los datos elegidos para el proyecto tienen problemas de multiíndice y filas con celdas vacías, se realizará una limpieza de datos para poder trabajar con ellos.

## Objetivo de la remodelación y limpieza de datos.

Lo que se observa en los conjuntos de datos que se generaron para cada año es que aún los nombres de las columnas se mantienen como unnamed. Lo que queremos conseguir es que cada columna sea una variable y cada obseración sea una fila. El resultado al que se quiere llegar es el siguiente:

| Nombre de la variables  | Descripción     |
| :-------------: | :------------- |
| categoria_principal | Categoría principal de la enfermedad |
| categoria_nivel1 | Subcategoría |
| categoria_nivel2 | Subsubcategoría (si la hay) |
| nombre_enfermedad | Nombre de la enfermedad. Es el nivel más bajo dentro de las categorías de la OMS.|
| codigo| Código según la OMS. |
| ano | Año correspondiente de la hoja. |
| sexo | Sexo de la persona afectada. |
| edad | Rango de edad.|
| dalys | Valor numérico de los DALYs para la enfermedad. |

**Tabla 1:** Objetivo de la remodelación y limpieza de datos. Se busca que cada columna sea una variable y cada fila una observación.

Por lo tanto, el siguiente paso será remodelar los conjuntos de datos por año para conseguir un formato adecuado para el análisis posterior.

In [19]:
# Librerías necesarias
import pandas as pd
import numpy as np

In [ ]:
# Definimos el path del archivo .xslsx y los años que nos interesan para el análisis.
archivo = 'data_raw/datos_analisis/DALYs/ghe2021_daly_global_new.xlsx'

# Cargamos datos y pandas.
df_2021 = pd.read_excel(archivo, sheet_name="Global 2021")
df_2021

In [59]:
# Revisamos cómo se ven los datos de 2021 para identificar la filas que no sirven.
print(df_2021.head(10))

  Global Health Estimates 2021: DALYs by age, sex and cause      Unnamed: 1  \
0                                                NaN                    NaN   
1                                            Region:                    NaN   
2                                              Year:                    NaN   
3                                                NaN                    NaN   
4                                                NaN                    NaN   
5                                                NaN                    NaN   
6                             Population (thousands)                    NaN   
7                                               Code         Cause of death   
8                                                  0                    NaN   
9                                                 10                     I.   

                                          Unnamed: 2 Unnamed: 3 Unnamed: 4  \
0                                                NaN

Después de cargar los datos, se observa que las primeras 6 filas no contienen datos útiles, por lo que se eliminarán. Además, los encabezados están en la fila 5 (index 4), por lo que se renombrarán las columnas con los valores de esa fila. El resultado de esta limpieza se guardará en un nuevo archivo Excel para cada año.

In [60]:
# Asignamos bajo la variable "nombres_columnas" los valores de la fila 5 (index 4) para renombrar las columnas.
nombres_columnas = df_2021.iloc[5].tolist()
print(nombres_columnas)

[nan, nan, nan, nan, nan, 'Age group', 'Total (all ages)', nan, nan, '0-28 days', '1-59 months', '5-14', '15-29', '30-49', '50-59', '60-69', '70+', '0-28 days', '1-59 months', '5-14', '15-29', '30-49', '50-59', '60-69', '70+']


In [61]:
# Dado que las primeras 6 filas no contienen datos útiles, las eliminamos y renombramos las columnas. Esto es, los encabezados están en la fila 5 (index 4)
df_2021.columns = nombres_columnas
df_2021 = df_2021.iloc[6:].reset_index(drop=True)
print(df_2021.head(5))

                      NaN             NaN  \
0  Population (thousands)             NaN   
1                    Code  Cause of death   
2                       0             NaN   
3                      10              I.   
4                      20             NaN   

                                                 NaN  \
0                                                NaN   
1                                                NaN   
2                                         All Causes   
3  Communicable, maternal, perinatal and nutritio...   
4                                                 A.   

                                 NaN  NaN Age group  Total (all ages)  \
0                                NaN  NaN       NaN       7939383.133   
1                                NaN  NaN       NaN               NaN   
2                                NaN  NaN       NaN  3013734614.04409   
3                                NaN  NaN       NaN  928759077.469129   
4  Infectious and parasitic

In [62]:
# Se utiliza la función "ffill" para rellenar los valores faltantes en las columnas de jerarquía (las primeras 4 columnas) con el último valor no nulo encontrado hacia adelante. Esto es necesario porque en el archivo original, los valores de estas columnas solo se registran una vez para cada grupo, y las filas siguientes dentro del mismo grupo tienen valores nulos.
# Por cada una de las primeras 4 columnas, se aplica ffill para que se rellenen los calores del último valor no nulo hacia adelante.
# Esto asegura que cada fila tenga la información completa de jerarquía.

columnas_jerarquia = df_2021.columns[:4].tolist()
for col in columnas_jerarquia:
    df_2021[col] = df_2021[col].ffill()

C:\Users\troop\AppData\Local\Temp\ipykernel_20196\2461595720.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_2021[col] = df_2021[col].ffill()


In [63]:
# Por cada columna i dada en la lista df_2021.columns se enumera su índice. 
# Luego, si el nombre es 'Causa', se renombra como 'Causa_i', donde i es el índice de la columna. 
# Si el nombre de la columna no es 'Causa' y no es nulo, se mantiene el mismo nombre. 
# Si el nombre de la columna es nulo, se renombra como 'col_i', donde i es el índice de la columna.

nuevos_nombres = []
for i, col in enumerate(df_2021.columns):
    if col == 'Causa':
        nuevos_nombres.append(f'Causa_{i}')
    else:
        nuevos_nombres.append(col if pd.notna(col) else f'col_{i}')

df_2021.columns = nuevos_nombres

In [64]:
# Renombramos las columnas [0, 1, 2, 3, 4] con nombres más descriptivos.

df_2021 = df_2021.rename(columns={
    df_2021.columns[0]: 'codigo',
    df_2021.columns[1]: 'categoria_principal',
    df_2021.columns[2]: 'categoria_nivel1',
    df_2021.columns[3]: 'categoria_nivel2',
    df_2021.columns[4]: 'causa'
})

In [65]:
# Igualmente, dado que no nos interesan las columnas correspondientes al total de DALYs por grupo etario para mujeres, hombres y ambos sexos, las eliminamos.
columnas_eliminar = [6, 7, 8]
df_2021 = df_2021.drop(columns=df_2021.columns[columnas_eliminar])

In [66]:
# Primero separamos as filas correspondientes a población
# Separamos la fila de población del resto de los datos por si acaso.
poblacion = df_2021['codigo'] == 'Population (thousands)'
poblacion_df = df_2021[poblacion].copy() # Copiamos el dataset sin la fila de población para trabajar con él.

# Definimos el dataset de enfermedades, se acuerdo con que no incluya la fila de población.
df_enfermedades = df_2021[~poblacion].copy()


In [67]:
# También eliminas dos filas de causas totales y una adicional del código.
# Conservamos las filas que no cumplan con el siguiente criterio:
# Para la primer fila...
filas_eliminar = df_enfermedades['categoria_principal'] != 'Cause of death'


# Para la segunda fila...
filas_eliminar2 = df_enfermedades['categoria_nivel2'] != 'All Causes'

# Ambos criterios a la vez...
filas_finales = filas_eliminar & filas_eliminar2

# Aplicamos el filtro al dataset de enfermedades.
df_enfermedades = df_enfermedades[filas_finales].copy()


En el caso de las enfermedades, se acomodará el dataset manualmente. Por lo tanto, generamos un nuevo dataset con las columnas necesarias para el merge posterior con los datos de DALYs organizados por edad y sexo. Estas columnas son: código, categoría principal, categoría nivel 1, categoría nivel 2, causa y Age group. 

In [68]:
# Lista de nombres de columnas de jerarquía para un nuevo conjunto de datos.
lista_enfermedades = ['codigo', 'categoria_principal', 'categoria_nivel1', 'categoria_nivel2', 'causa', 'Age group']

# Separamos las columnas correspondientes a enferemdades, junto con sus respectivas columnas de jerarquía.
df_jerarquias = df_enfermedades[lista_enfermedades].copy()

# Revisamos cómo se ven los datos de enfermedades después de la limpieza.
df_jerarquias.head(5)

,codigo,categoria_principal,categoria_nivel1,categoria_nivel2,causa,Age group
3,10,I.,"Communicable, maternal, perinatal and nutritio...",NaN,NaN,NaN
4,20,I.,A.,Infectious and parasitic diseases,NaN,NaN
5,30,I.,A.,1.,Tuberculosis,NaN
6,40,I.,A.,2.,STDs excluding HIV,NaN
7,50,I.,A.,2.,a.,Syphilis


Para reacomodar las variables sexo y edad, se realizará un conjunto de datos adicionales para hombre y mujer. El criterio para separarlos será los rangos de edad. Se mantendrá la columna de código para poder hacer un merge posterior.

In [70]:
# Lista de nombres de columnas de jerarquía para un nuevo conjunto de datos.
lista_edades = ['codigo', '0-28 days', '1-59 months', '5-14', '15-29', '30-49', '50-59', '60-69', '70+']

# Separamos las columnas correspondientes a enferemdades, junto con sus respectivas columnas de jerarquía.
df_dalys = df_enfermedades[lista_edades].copy()

# Revisamos cómo se ven los datos de enfermedades después de la limpieza.
df_dalys.head(5)

,codigo,0-28 days,0-28 days,1-59 months,1-59 months,5-14,5-14,15-29,15-29,30-49,30-49,50-59,50-59,60-69,60-69,70+,70+
3,10,108619505.732124,87348406.31804,107049176.924865,94393221.570602,27194740.771673,27176421.277304,30027294.794567,40345408.474024,76959471.353592,62741726.416519,48103603.104897,31149814.523773,51640075.468003,35087909.205697,53075557.158055,47846742.394623
4,20,5591090.915649,4659805.488572,64964988.33156,59058201.502689,15062423.457068,14635737.197366,15322252.485276,14471883.215072,30067654.390141,21671120.836294,13367848.538517,9712354.848535,9683874.422252,8049872.441309,8672584.539365,9888397.276761
5,30,33.046979,54.595532,4932044.606369,5576673.885822,1094008.747922,1458158.256437,4368796.959188,3330256.044898,10186358.897613,5390537.897352,6338681.258957,3481570.464763,4950131.654464,2912587.56063,3757638.327138,2858887.185742
6,40,2079916.572509,1869392.706819,1227267.274708,735920.579723,38893.844012,37304.795356,150278.020065,223440.796895,258188.750058,482291.741925,59114.981126,118600.470944,27037.423189,58295.805218,19681.292545,46001.979003
7,50,2079916.572509,1869392.706819,1227267.274708,735920.579723,34473.104943,32463.039843,22189.817935,19973.791919,38787.913406,19925.620285,15750.984955,10533.138779,12767.311972,6054.660962,11533.322558,5797.861265


Como el conjunto resultante de este proceso son todos los rangos de edad acomodados como 'hombre' y después el rango correspondiente a 'mujer', se aplicará una función para clasificar los grupos etarios.

In [ ]:
# Definimos los correspondientes a hombres de acuerdo con su posición.
index_hombres = [1, 3, 5, 7, 9, 11, 13, 15]

# Columnas de hombres
df_dalys_hombres = df_dalys.iloc[:, index_hombres].copy()


# Agregar la columna 'codigo' (índice 0)
df_dalys_hombres.insert(0, 'codigo', df_dalys['codigo'])

# Columnas mujeres
# Definimos los correspondientes a mujeres de acuerdo con su posición.
index_mujeres = [2, 4, 6, 8, 10, 12, 14, 16]

# Columnas de hombres
df_dalys_mujeres = df_dalys.iloc[:, index_mujeres].copy()

# Agregar la columna 'codigo' (índice 0)
df_dalys_mujeres.insert(0, 'codigo', df_dalys['codigo'])

In [99]:
# Verificamos si se seleccionaron dalys de hombres.
print(df_dalys_hombres)

     codigo       edad             dalys
0        10  0-28 days  108619505.732124
1        20  0-28 days    5591090.915649
2        30  0-28 days         33.046979
3        40  0-28 days    2079916.572509
4        50  0-28 days    2079916.572509
...     ...        ...               ...
1707   1600        70+    1657326.433013
1708   1610        70+    1225272.955303
1709   1620        70+     398699.994156
1710   1630        70+      33353.483554
1711   1700        70+    7155417.886174

[1712 rows x 3 columns]


In [126]:
# Verificamos si se seleccionaron dalys de mujeres.
print(df_dalys_mujeres)

     codigo       edad           dalys   sexo
0        10  0-28 days  87348406.31804  mujer
1        20  0-28 days  4659805.488572  mujer
2        30  0-28 days       54.595532  mujer
3        40  0-28 days  1869392.706819  mujer
4        50  0-28 days  1869392.706819  mujer
...     ...        ...             ...    ...
1707   1600        70+  1038012.176584  mujer
1708   1610        70+    742162.82966  mujer
1709   1620        70+   265700.091701  mujer
1710   1630        70+    30149.255223  mujer
1711   1700        70+   5568120.53372  mujer

[1712 rows x 4 columns]


In [ ]:
# Hacemos un pd.melt para el dataframe de hombres.

# Definimos las columnas nuevas que se convertirán en columnas de variable y valor.
columnas_edad = ['0-28 days', '1-59 months', '5-14', '15-29', '30-49', '50-59', '60-69', '70+']

df_dalys_hombres = pd.melt(
    df_dalys_hombres,
    id_vars=['codigo'],
    value_vars=columnas_edad,
    var_name='edad',
    value_name='dalys'
)

# Hacemos un pd.melt para el dataframe de mujeres.

df_dalys_mujeres = pd.melt(
    df_dalys_mujeres,
    id_vars=['codigo'],
    value_vars=columnas_edad,
    var_name='edad',
    value_name='dalys'
)


ValueError: value_name (dalys) cannot match an element in the DataFrame columns.

Ahora que se tienen dos dataframes independientes de hombres y mujeres, agregamos una nueva columna con la variable sexo. Y posteriormente se hace un merge deacuerdo con la columna 'codigo' como índice. 

In [123]:
# Agregamos la variable de sexo con su valor respectivo de acuerdo con el dataframe.
df_dalys_hombres['sexo'] = 'hombre'
df_dalys_mujeres['sexo'] = 'mujer'

# Hacemos el merge de acuerdo con el índice 'codigo'
df_dalys_completo = pd.merge(df_dalys_hombres, df_dalys_mujeres, how='outer', on=['codigo', 'sexo', 'edad', 'dalys'])

In [ ]:
# Revisamos que los datos estén acomodados como queríamos.
print(df_dalys_completo)

     codigo         edad             dalys    sexo
0        10    0-28 days  108619505.732124  hombre
1        10  1-59 months  107049176.924865  hombre
2        10        15-29   30027294.794567  hombre
3        10        30-49   76959471.353592  hombre
4        10         5-14   27194740.771673  hombre
...     ...          ...               ...     ...
3419   1700        30-49    3879304.319166   mujer
3420   1700         5-14     133021.747245   mujer
3421   1700        50-59    3210947.613694   mujer
3422   1700        60-69    4545445.842791   mujer
3423   1700          70+     5568120.53372   mujer

[3424 rows x 4 columns]


In [134]:
# Definimos el path del archivo .xslsx ya editado manualmente
archivo = 'data_raw/DALY_def/DALY_enfermedades2021.xlsx'

# Cargamos datos bajo el nombre df_enfermedades_2021
df_enfermedades_2021 = pd.read_excel(archivo, sheet_name="Global 2021")
df_enfermedades_2021

,codigo,categoria_principal,categoria_nivel1,categoria_nivel2,causa
0,10,"Communicable, maternal, perinatal and nutritio...",NaN,NaN,NaN
1,20,I.,Infectious and parasitic diseases,NaN,NaN
2,30,I.,A.,1.,Tuberculosis
3,40,I.,A.,STDs excluding HIV,NaN
4,50,I.,A.,2.,Syphilis
...,...,...,...,...,...
209,1600,III.,B.,Intentional injuries,NaN
210,1610,III.,B.,1.,Self-harm
211,1620,III.,B.,2.,Interpersonal violence
212,1630,III.,B.,3.,Collective violence and legal intervention


In [138]:
# Una vez que terminamos de rearreglar los datos manualmente, haremos un merge entre df_dalys_completo con df_enfermedades.
# Se hizo una copia del datset df_enfermedades.
# Igualmente haremos el merge de acuerdo con el índice 'codigo'
df_total = pd.merge(df_dalys_completo, df_enfermedades_2021, how='outer', on='codigo')

# Agregamos la dos variables de país y año.
df_total['ano'] = '2021'
df_total['pais'] = 'global'

print(df_total)

     codigo         edad             dalys    sexo  \
0        10    0-28 days  108619505.732124  hombre   
1        10  1-59 months  107049176.924865  hombre   
2        10        15-29   30027294.794567  hombre   
3        10        30-49   76959471.353592  hombre   
4        10         5-14   27194740.771673  hombre   
...     ...          ...               ...     ...   
3419   1700        30-49    3879304.319166   mujer   
3420   1700         5-14     133021.747245   mujer   
3421   1700        50-59    3210947.613694   mujer   
3422   1700        60-69    4545445.842791   mujer   
3423   1700          70+     5568120.53372   mujer   

                                    categoria_principal categoria_nivel1  \
0     Communicable, maternal, perinatal and nutritio...              NaN   
1     Communicable, maternal, perinatal and nutritio...              NaN   
2     Communicable, maternal, perinatal and nutritio...              NaN   
3     Communicable, maternal, perinatal and nut

# Para pruebas de visulaización de datos...

In [ ]:
# Conjunto de la hoja de 2021
df_2021.to_excel('data_raw/DALY_def/DALY_2021.xlsx', index=False, sheet_name='Global 2021')


In [ ]:
# Conjunto de datos de jerarquía
df_jerarquias.to_excel('data_raw/DALY_def/DALY_jerarquias2021.xlsx', index=False, sheet_name='Global 2021')


In [71]:
# Conjunto de datos de dalys
df_dalys.to_excel('data_raw/DALY_def/DALY_dalys2021.xlsx', index=False, sheet_name='Global 2021')


In [ ]:
# Conjunto de datos de dalys de edad y sexo.
df_dalys_completo.to_excel('data_raw/DALY_def/DALY_dalys_completo2021.xlsx', index=False, sheet_name='Global 2021')

In [139]:
# Conjunto de datos limpio de dalys en formato tidy data
df_total.to_excel('data_raw/DALY_def/DALY_dalys_total2021.xlsx', index=False, sheet_name='Global 2021')